# Canada City Selection

This notebook selects a representative set of Canadian cities from the World Cities dataset.
Capital and provincial cities are always included; additional cities are added only if they
are at least `MIN_SPACING_M` metres apart from any already-selected city.

**Workflow:**
1. Load raw world-cities data
2. Prepare & clean (filter country, build GeoDataFrame, reproject)
3. Select cities (capitals + spacing-based algorithm)
4. Export to processed folder

## 0  |  Imports & Constants

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

# ── File paths ───────────────────────────────────────────────────────────────
# pathlib keeps paths portable across Windows, macOS, and Linux.
# The notebook lives in notebooks/, so we step up one level to the project root.
DATA_DIR       = Path("../data")
RAW_PATH       = DATA_DIR / "raw"       / "worldcities.csv"
PROCESSED_PATH = DATA_DIR / "processed" / "selected_cities_canada.csv"

# ── Coordinate reference systems ─────────────────────────────────────────────
# EPSG:4326  — geographic (degrees), used by the source CSV.
# EPSG:3347  — Statistics Canada Lambert, metric projection for Canada;
#              required for distance calculations in metres.
CRS_GEOGRAPHIC = "EPSG:4326"
CRS_METRIC     = "EPSG:3347"

# ── Analysis parameters ──────────────────────────────────────────────────────
TARGET_COUNTRY = "Canada"

# Capital types that are always included regardless of spacing.
CAPITAL_TYPES = ["primary", "admin"]

# Minimum allowed distance between any two selected cities (metres).
# Cities closer than this are skipped to avoid spatial clustering.
MIN_SPACING_M = 150_000

# Maximum total cities to select (capitals + additional).
MAX_CITIES = 41

## 1  |  Load Raw Data

In [3]:
# Fail fast: verify the file exists before any heavy processing.
if not RAW_PATH.is_file():
    raise FileNotFoundError(f"Raw data not found: {RAW_PATH.absolute()}")

cities_df = pd.read_csv(RAW_PATH)

assert len(cities_df) > 0, "Loaded file is empty — check the file path."
print(f"Loaded {len(cities_df):,} cities worldwide.")

Loaded 48,059 cities worldwide.


## 2  |  Prepare & Clean

In [4]:
# ── 2a  Filter to target country ─────────────────────────────────────────────
canada_df = cities_df[cities_df["country"] == TARGET_COUNTRY].copy()

assert len(canada_df) > 0, f"No cities found for country '{TARGET_COUNTRY}'."
print(f"{TARGET_COUNTRY} cities in raw data: {len(canada_df):,}")

# ── 2b  Check required columns are present ───────────────────────────────────
required_columns = ["lat", "lng", "capital", "population"]
for col in required_columns:
    if col not in canada_df.columns:
        raise KeyError(f"Required column missing from source data: '{col}'")

# ── 2c  Build GeoDataFrame in geographic CRS ─────────────────────────────────
canada_gdf = gpd.GeoDataFrame(
    canada_df,
    geometry=gpd.points_from_xy(canada_df["lng"], canada_df["lat"]),
    crs=CRS_GEOGRAPHIC,
)

# ── 2d  Reproject to metric CRS for distance calculations ────────────────────
# Distance calculations on geographic coordinates (degrees) are meaningless;
# EPSG:3347 gives us metres across the full extent of Canada.
canada_gdf = canada_gdf.to_crs(CRS_METRIC)

assert canada_gdf.crs is not None, "Reprojection failed — CRS is undefined."
print(f"GeoDataFrame CRS: {canada_gdf.crs.to_epsg()}")

Canada cities in raw data: 485
GeoDataFrame CRS: 3347


## 3  |  Select Cities

In [5]:
def select_cities_by_spacing(
    seed_gdf: gpd.GeoDataFrame,
    candidates_gdf: gpd.GeoDataFrame,
    min_spacing_m: float,
    max_cities: int,
) -> gpd.GeoDataFrame:
    """
    Adds candidate cities to a seed selection if they are sufficiently spaced.

    Iterates through candidates (largest population first) and includes each
    city only when its distance to every already-selected city exceeds
    `min_spacing_m`. Stops early once `max_cities` is reached.

    Parameters
    ----------
    seed_gdf : gpd.GeoDataFrame
        Cities that are always included (e.g. capitals). Must share the
        same CRS as `candidates_gdf`.
    candidates_gdf : gpd.GeoDataFrame
        Remaining cities to evaluate, sorted by descending population.
    min_spacing_m : float
        Minimum distance in metres between any two selected cities.
    max_cities : int
        Hard upper limit on the total number of selected cities.

    Returns
    -------
    gpd.GeoDataFrame
        Combined GeoDataFrame of seed cities plus accepted candidates.
    """
    if not seed_gdf.crs == candidates_gdf.crs:
        raise ValueError(
            f"CRS mismatch: seed={seed_gdf.crs.to_epsg()}, "
            f"candidates={candidates_gdf.crs.to_epsg()}"
        )

    selected_gdf = seed_gdf.copy()

    for _, city in candidates_gdf.iterrows():
        if len(selected_gdf) >= max_cities:
            break

        min_distance_m = selected_gdf.distance(city.geometry).min()

        if min_distance_m >= min_spacing_m:
            selected_gdf = pd.concat(
                [selected_gdf, city.to_frame().T],
                ignore_index=True,
            )

    return selected_gdf


# ── 3a  Split into capitals and all other cities ─────────────────────────────
capital_cities_gdf = canada_gdf[canada_gdf["capital"].isin(CAPITAL_TYPES)]

# Non-capital cities sorted largest-first so the most populated places are
# considered before smaller towns when filling the remaining slots.
other_cities_gdf = (
    canada_gdf[~canada_gdf.index.isin(capital_cities_gdf.index)]
    .sort_values("population", ascending=False)
    .reset_index(drop=True)
)

assert len(capital_cities_gdf) > 0, "No capital or admin cities found — check the 'capital' column values."
print(f"Seed cities (capitals/admin): {len(capital_cities_gdf):,}")

# ── 3b  Run spacing-based selection ──────────────────────────────────────────
selected_cities_gdf = select_cities_by_spacing(
    seed_gdf=capital_cities_gdf,
    candidates_gdf=other_cities_gdf,
    min_spacing_m=MIN_SPACING_M,
    max_cities=MAX_CITIES,
)

assert len(selected_cities_gdf) > 0, "Selection returned an empty dataset."
print(f"Total cities selected: {len(selected_cities_gdf):,} (cap: {MAX_CITIES})")

Seed cities (capitals/admin): 14
Total cities selected: 41 (cap: 41)


## 4  |  Export

In [6]:
# Create the output directory if it does not yet exist (safe on re-runs).
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

selected_cities_gdf.to_csv(PROCESSED_PATH, index=False)

print(f"Saved {len(selected_cities_gdf):,} cities → {PROCESSED_PATH}")

Saved 41 cities → ..\data\processed\selected_cities_canada.csv
